# Hands-On Lab: Governed RAG, Provenance Verification & Sandboxed Multi-Agent Workflows
**Companion Lab to the GFDRR 3-Tier Staff AI-Upskilling Playbook**

This notebook runs 100% offline in `deterministic_local` mode and walks staff through:
1. **Track 1 (Foundational):** Testing World Bank AMS 6.21A classification gates, inspecting 3-layer PII redaction, and catching a post-export tampered brief.
2. **Track 2 (Applied):** Recording M&E specialist HITL chunk relevance judgments in SQLite and measuring out-of-fold `nDCG@5` uplift.
3. **Track 3 (Technical):** Enforcing least-privilege agent capability scopes, blocking prompt-injection/escalation attacks, recovering from mid-run worker crashes via SQLite checkpoints, and signing a Human-in-Command (HIC) release.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    ROOT = ROOT.parent.parent.parent

for rel in ('projects/gfdrr_corpus', 'projects/08-gfdrr-me-governed-rag/src', 'projects/09-gfdrr-sandboxed-drm-agent/src'):
    p = str((ROOT / rel).resolve())
    if p not in sys.path:
        sys.path.insert(0, p)

from gfdrr_rag.pipeline import GovernedRAGPipeline
from gfdrr_rag.eval.hitl_loop import run_hitl_feedback_experiment
from gfdrr_agent.orchestrator.supervisor import SupervisorConfig, SupervisorOrchestrator, SimulatedWorkerCrashError
from gfdrr_agent.security.sandbox import SecurityViolationError

rag = GovernedRAGPipeline()
print(f'Loaded {len(rag.chunks)} section-preserved chunks across {len(rag.parents)} GFDRR documents.')

## Track 1 Exercise: WB Classification Gate, PII Redaction & Tamper Detection

In [ ]:
# 1A. External user attempting to access a Confidential sovereign Cat DDO note -> Hard Blocked & Audited
blocked = rag.query(
    'Disclose Belize Cat DDO parametric wind-knot threshold in GFDRR-BZ-2026-005.',
    clearance_role='public_external',
    metadata_filter={'country': 'Belize'},
)
print('Classification Verdict:', blocked.classification_verdict.decision)
print('Answer:', blocked.synthesis.answer_text)

# 1B. Authorized M&E Specialist querying a field verification note with synthetic PII -> 0% Leakage
pii_resp = rag.query(
    'Report the Dadu Sindh field spot-check mortar strength result and grievance closure details under grant GFDRR-PK-2026-004.',
    clearance_role='me_specialist',
    metadata_filter={'country': 'Pakistan'},
)
print('\nRedactions by Checkpoint:', pii_resp.checkpoint_redaction_counts)
print('Zero PII Leakage Verified:', pii_resp.zero_pii_leakage_verified)
print(pii_resp.synthesis.answer_text)

# 1C. Export signed brief and test tamper detection
brief_md = rag.export_brief(pii_resp, title='Pakistan Resilient Housing Verification')
print('\nIntact Verification:', rag.verify_brief(brief_md).status)
tampered_md = brief_md.replace('4.8 MPa', '9.2 MPa')
print('After 1-Number Edit:', rag.verify_brief(tampered_md).reason)

## Track 2 Exercise: Continuous HITL Relevance Labeling & Weight Tuning

In [ ]:
hitl_stats = run_hitl_feedback_experiment(pipeline=rag)
print(f"Evaluated {hitl_stats['n_chunk_judgments']} chunk judgments across {hitl_stats['n_queries_evaluated']} answerable queries:")
print(f"  Untuned nDCG@5 : {hitl_stats['untuned_ndcg_at_5']:.3f}")
print(f"  HITL-Tuned nDCG@5 (4-Fold OOF): {hitl_stats['tuned_ndcg_at_5']:.3f} (Δ = +{hitl_stats['delta_ndcg_at_5']:.3f}, 95% CI {hitl_stats['ndcg_ci95']})")
print(f"  Human vs Automated Judge Quadratic Cohen's κ_w: {hitl_stats['cohen_kappa_quadratic']:.3f}")

## Track 3 Exercise: Sandboxed Agent Scopes, Crash Recovery & Mandatory HIC Gate

In [ ]:
sup = SupervisorOrchestrator(SupervisorConfig(enable_critic=True, require_hic_approval=True, inject_unverified_claim_for_audit=True), rag_pipeline=rag)

# 3A. Attempt unauthorized tier escalation from Grant M&E Analyst
try:
    sup.sandbox.invoke(
        run_id='LAB-SEC-01',
        agent_role='grant_me_analyst',
        tool_name='rag.query',
        params={'query': 'Haiti EOC depot', 'clearance_role': 'restricted_audit', 'requested_tier': 'Strictly Confidential'},
    )
except SecurityViolationError as err:
    print('Sandbox Blocked Escalation:', err.violation_type, '->', err)

# 3B. Run full workflow -> Pause at HIC Gate -> Submit signed HIC approval
paused = sup.execute_brief('Synthesize Nature-based Solutions flood retention and exposure across Pakistan and Viet Nam.')
print('\nWorkflow Status Before Human Sign-off:', paused.status, '| Unsupported Claims Caught by Critic:', paused.unsupported_claims_caught)
approved = sup.submit_hic_decision(
    run_id=paused.run_id,
    reviewer_id='drm_lead_lab',
    reviewer_role='GFDRR Senior DRM Specialist',
    decision='approve',
    rationale='Verified NbS wetland & mangrove financing figures and Admin-1 flood exposure alignment.',
)
print('Final Status:', approved.status, '| HIC HMAC Prefix:', approved.hic_record.signature_hmac[:24])